In [1]:
from pathlib import Path

root = Path("/kaggle/input")

for filename in [
    "record_202.npz",
    "cnn_rr_fp32.onnx",
    "normalization.json",
    "ds2_predictions.csv",
]:
    matches = sorted(root.rglob(filename))
    print(f"\n{filename}:")
    if matches:
        for path in matches:
            print(path)
    else:
        print("Chưa tìm thấy")


record_202.npz:
/kaggle/input/datasets/quanghuy10/record-202/record_202.npz

cnn_rr_fp32.onnx:
/kaggle/input/datasets/quanghuy10/tv2-results/exports/cnn_rr_fp32.onnx

normalization.json:
/kaggle/input/datasets/quanghuy10/tv2-results/normalization.json

ds2_predictions.csv:
/kaggle/input/datasets/quanghuy10/tv2-results/results/ds2_predictions.csv


In [2]:
%pip install -q onnxruntime==1.30.0

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
from sklearn.metrics import confusion_matrix

DATA_202 = Path(
    "/kaggle/input/datasets/quanghuy10/record-202/record_202.npz"
)

RESULT_DIR = Path(
    "/kaggle/input/datasets/quanghuy10/tv2-results"
)

MODEL_PATH = RESULT_DIR / "exports/cnn_rr_fp32.onnx"
NORM_PATH = RESULT_DIR / "normalization.json"
PRED_PATH = RESULT_DIR / "results/ds2_predictions.csv"

OUT_DIR = Path("/kaggle/working/TV2_202_COMPARISON")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["N", "S", "V", "F", "Q"]

with DATA_202.open("rb") as stream:
    source_hash = hashlib.file_digest(stream, "sha256").hexdigest()

with np.load(DATA_202, allow_pickle=False) as source:
    data = {key: source[key] for key in source.files}

n = len(data["y"])

assert n > 0
assert data["X_waveform"].shape == (n, 1, 234)
assert data["X_rr"].shape == (n, 3)
assert data["y"].shape == (n,)
assert np.isfinite(data["X_waveform"]).all()
assert np.isfinite(data["X_rr"]).all()
assert (data["X_rr"] > 0).all()
assert np.isin(data["y"], np.arange(5)).all()
assert set(data["record_id"].astype(str)) == {"202"}
assert set(data["patient_id"].astype(str)) == {"P_201"}
assert np.all(np.diff(data["r_peak_index"]) > 0)

normalization = json.loads(
    NORM_PATH.read_text(encoding="utf-8")
)

assert float(normalization["ecg_std"]) > 0
assert np.all(np.asarray(normalization["rr_std"]) > 0)

ecg = np.ascontiguousarray(
    (
        data["X_waveform"] - normalization["ecg_mean"]
    ) / normalization["ecg_std"],
    dtype=np.float32,
)

rr = np.ascontiguousarray(
    (
        data["X_rr"]
        - np.asarray(normalization["rr_mean"], dtype=np.float64)
    ) / np.asarray(normalization["rr_std"], dtype=np.float64),
    dtype=np.float32,
)

all_predictions = pd.read_csv(
    PRED_PATH,
    dtype={"record_id": str, "patient_id": str},
)

seed_values = pd.to_numeric(
    all_predictions["seed"], errors="raise"
)

ds2 = all_predictions.loc[
    all_predictions["model_variant"].eq("cnn_rr")
    & seed_values.eq(42)
].copy()

assert len(ds2) == 47557, (
    f"Cần 47.557 dòng CNN + RR seed 42, nhận được {len(ds2)}."
)
assert "202" not in set(ds2["record_id"])
assert ds2["record_id"].nunique() == 21
assert not ds2.duplicated(
    ["record_id", "r_peak_index"]
).any()

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(
        y_true, y_pred, labels=np.arange(5)
    ).astype(np.float64)

    tp = np.diag(cm)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)

    se = np.divide(
        tp,
        support,
        out=np.full(5, np.nan),
        where=support > 0,
    )

    ppv = np.divide(
        tp,
        predicted,
        out=np.full(5, np.nan),
        where=predicted > 0,
    )

    denominator = support + predicted

    f1 = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(5),
        where=denominator > 0,
    )

    return {
        "se_S": se[1],
        "ppv_S": ppv[1],
        "se_V": se[2],
        "ppv_V": ppv[2],
        "f1_macro": f1.mean(),
        "accuracy": tp.sum() / cm.sum(),
    }

original_metrics = calculate_metrics(
    ds2["y_true"].to_numpy(dtype=np.int64),
    ds2["y_pred"].to_numpy(dtype=np.int64),
)

assert np.isclose(
    original_metrics["f1_macro"], 0.488642, atol=1e-6
), "F1 DS2 gốc chưa khớp kết quả đã bàn giao."

assert np.isclose(
    original_metrics["accuracy"], 0.855521, atol=1e-6
), "Accuracy DS2 gốc chưa khớp kết quả đã bàn giao."

session = ort.InferenceSession(
    str(MODEL_PATH),
    providers=["CPUExecutionProvider"],
)

probability_parts = []

for start in range(0, n, 256):
    end = min(start + 256, n)

    output = session.run(
        ["probabilities"],
        {
            "ecg": ecg[start:end],
            "rr": rr[start:end],
        },
    )[0]

    probability_parts.append(output)

probabilities = np.concatenate(probability_parts, axis=0)

assert probabilities.shape == (n, 5)
assert np.isfinite(probabilities).all()
assert np.allclose(
    probabilities.sum(axis=1), 1.0, atol=1e-5
)

predictions_202 = probabilities.argmax(axis=1)

record_predictions = pd.DataFrame({
    "model_variant": "cnn_rr",
    "seed": 42,
    "sample_index": np.arange(n),
    "record_id": data["record_id"].astype(str),
    "patient_id": data["patient_id"].astype(str),
    "r_peak_index": data["r_peak_index"],
    "y_true": data["y"],
    "y_pred": predictions_202,
})

for position, name in enumerate(CLASS_NAMES):
    record_predictions[f"p_{name}"] = probabilities[:, position]

combined_true = np.concatenate([
    ds2["y_true"].to_numpy(dtype=np.int64),
    data["y"],
])

combined_pred = np.concatenate([
    ds2["y_pred"].to_numpy(dtype=np.int64),
    predictions_202,
])

combined_metrics = calculate_metrics(
    combined_true, combined_pred
)

comparison = pd.DataFrame([
    {
        "evaluation_set": "DS2_without_202",
        "n_beats": len(ds2),
        "n_records": 21,
        **original_metrics,
    },
    {
        "evaluation_set": "DS2_with_202",
        "n_beats": len(combined_true),
        "n_records": 22,
        **combined_metrics,
    },
])

metric_columns = [
    "se_S", "ppv_S", "se_V", "ppv_V",
    "f1_macro", "accuracy",
]

delta = pd.DataFrame([{
    metric: combined_metrics[metric] - original_metrics[metric]
    for metric in metric_columns
}])

comparison.to_csv(
    OUT_DIR / "ds2_with_without_202.csv",
    index=False,
    encoding="utf-8-sig",
)

delta.to_csv(
    OUT_DIR / "delta_with_minus_without_202.csv",
    index=False,
    encoding="utf-8-sig",
)

record_predictions.to_csv(
    OUT_DIR / "record_202_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "model_variant": "cnn_rr",
    "training_seed": 42,
    "model_path": str(MODEL_PATH),
    "normalization_path": str(NORM_PATH),
    "original_predictions_path": str(PRED_PATH),
    "record_202_path": str(DATA_202),
    "record_202_sha256": source_hash,
    "class_names": CLASS_NAMES,
    "normalization_refitted": False,
    "retrained": False,
    "main_result": "DS2_without_202",
    "supplementary_result": "DS2_with_202",
    "patient_overlap_note": (
        "Record 202 shares patient P_201 with "
        "training record 201."
    ),
    "original_ds2_predictions": "Saved PyTorch FP32 predictions",
    "record_202_predictions": "ONNX FP32 probabilities with argmax",
}

(OUT_DIR / "evaluation_config.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Mô hình: CNN + RR | Seed huấn luyện: 42 | ONNX FP32")
print("DS2 gốc đã khớp kết quả bàn giao.")
print("Số nhịp bản ghi 202:", n)

print("\nBẢNG SO SÁNH — THANG 0 ĐẾN 1:")
print(comparison.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\nCHÊNH LỆCH CÓ 202 TRỪ KHÔNG CÓ 202:")
print(delta.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\nĐã lưu kết quả tại:", OUT_DIR)
print("DS2 không có 202 vẫn là kết quả chính.")
print("Bản có 202 chỉ dùng cho phân tích bổ sung vì trùng bệnh nhân.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
Mô hình: CNN + RR | Seed huấn luyện: 42 | ONNX FP32
DS2 gốc đã khớp kết quả bàn giao.
Số nhịp bản ghi 202: 2135

BẢNG SO SÁNH — THANG 0 ĐẾN 1:
 evaluation_set  n_beats  n_records     se_S    ppv_S     se_V    ppv_V  f1_macro  accuracy
DS2_without_202    47557         21 0.855780 0.336942 0.839425 0.923685  0.488642  0.855521
   DS2_with_202    49692         22 0.849211 0.326223 0.840062 0.902870  0.479780  0.846132

CHÊNH LỆCH CÓ 202 TRỪ KHÔNG CÓ 202:
     se_S     ppv_S     se_V     ppv_V  f1_macro  accuracy
-0.006569 -0.010719 0.000637 -0.020815 -0.008862 -0.009389

Đã lưu kết quả tại: /kaggle/working/TV2_202_COMPARISON
DS2 không có 202 vẫn là kết quả chính.
Bản có 202 chỉ dùng cho phân tích bổ sung vì trùng bệnh nhân.


In [4]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

result_dir = Path("/kaggle/working/TV2_202_COMPARISON")

shutil.make_archive(
    "/kaggle/working/TV2_202_COMPARISON",
    "zip",
    root_dir=result_dir,
)

display(FileLink("TV2_202_COMPARISON.zip"))

/kaggle/working/TV2_202_COMPARISON.zip